# Match-level timeseries across samples

This notebook plots matched peak intensities across samples at three granularity levels:

| Level        | Column                    | Description                                    |
| ------------ | ------------------------- | ---------------------------------------------- |
| **Compound** | `target_compound_formula` | Groups all ions belonging to the same compound |
| **Ion**      | `target_ion_formula`      | Groups isotopes of the same ion                |
| **Isotope**  | `target_isotope_formula`  | One-to-one with peaks matched to isotopes      |

Each section aggregates peak heights per sample at the chosen level and plots them over time.


### Load peaks


In [ ]:
import plotly.express as px
import plotly.io as pio

from mascope_sdk import MascopeClient


pio.templates.default = "plotly_dark"  # or "plotly_white"

mascope = MascopeClient()

# Load peaks across batches (adjust workspace/batches to your data)
peaks = mascope.load_peaks(workspace="My Workspace", batches="My Batch")

In [ ]:
# Keep only matched peaks
matched = peaks[peaks["target_compound_formula"].notna()].copy()
matched

In [ ]:
# Optional: narrow down to specific compounds, ions, or isotopes
# matched = matched[matched["target_compound_name"].isin(["Acetone", "Methanol"])]
# matched = matched[matched["target_ion_formula"].isin(["C3H7O+", "CH5O+"])]

# See what's available at each level:
for col in ["target_compound_formula", "target_ion_formula", "target_isotope_formula"]:
    vals = matched[col].unique()
    print(f"{col}: {len(vals)} unique")

matched

### Compound-level

Aggregate all peaks belonging to the same compound.


In [ ]:
compound_ts = (
    matched.groupby(["datetime_utc", "sample_item_name", "target_compound_formula"])[
        "height"
    ]
    .sum()
    .reset_index()
    .sort_values("datetime_utc")
)

fig = px.scatter(
    compound_ts,
    x="datetime_utc",
    y="height",
    color="target_compound_formula",
    hover_data=["sample_item_name"],
    title="Compound-level intensities across samples",
)
fig.update_layout(xaxis_title="Datetime (UTC)", yaxis_title="Intensity [cps]")
fig.show()

### Ion-level

Aggregate isotopes belonging to the same ion.


In [ ]:
ion_ts = (
    matched.groupby(["datetime_utc", "sample_item_name", "target_ion_formula"])[
        "height"
    ]
    .sum()
    .reset_index()
    .sort_values("datetime_utc")
)

fig = px.scatter(
    ion_ts,
    x="datetime_utc",
    y="height",
    color="target_ion_formula",
    hover_data=["sample_item_name"],
    title="Ion-level intensities across samples",
)
fig.update_layout(xaxis_title="Datetime (UTC)", yaxis_title="Intensity [cps]")
fig.show()

### Isotope-level

Each isotope maps one-to-one to a peak, so this is the finest granularity.


In [ ]:
isotope_ts = (
    matched.groupby(["datetime_utc", "sample_item_name", "target_isotope_formula"])[
        "height"
    ]
    .sum()
    .reset_index()
    .sort_values("datetime_utc")
)

fig = px.scatter(
    isotope_ts,
    x="datetime_utc",
    y="height",
    color="target_isotope_formula",
    hover_data=["sample_item_name"],
    title="Isotope-level intensities across samples",
)
# Truncate long names to avoid legend overflow
max_name_len = 30
for trace in fig.data:
    if len(trace.name) > max_name_len:
        trace.name = trace.name[:max_name_len] + "…"

fig.update_layout(xaxis_title="Datetime (UTC)", yaxis_title="Intensity [cps]")
fig.show()

### Time-averaged view

Resample any of the above timeseries to a coarser time bin. This example uses the ion-level data.


In [ ]:
import pandas as pd


# Resample to a coarser time bin (e.g. "1h", "10min")
time_aggregation = "1h"

ion_ts_avg = (
    ion_ts.groupby(
        ["target_ion_formula", pd.Grouper(key="datetime_utc", freq=time_aggregation)]
    )["height"]
    .mean()
    .reset_index()
    .dropna()
)

fig = px.scatter(
    ion_ts_avg,
    x="datetime_utc",
    y="height",
    color="target_ion_formula",
    title=f"Ion intensities - {time_aggregation} average",
)
fig.update_layout(xaxis_title="Datetime (UTC)", yaxis_title="Intensity (mean) [cps]")
fig.show()